# TEMPLATE 2: TEXT CLUSTERING & TOPIC MODELING (UNSUPERVISED)
### GEMASTIK DATA MINING & WEB MINING (SESUAI STUDI KASUS FASILKOM UI)
Pipeline ini dirancang khusus untuk kasus perlombaan di mana data teks **TIDAK MEMILIKI LABEL TARGET**. Tugasnya adalah mengelompokkan teks berita/artikel ke dalam klaster topik trending yang optimal dan dievaluasi menggunakan **Adjusted Rand Index (ARI)** atau **Silhouette Score**.

**Fitur Utama Pipeline:**
1. Text Preprocessing: pembersihan tag URL, normalisasi tanda baca, stopwords bahasa Indonesia.
2. Ekstraksi Vektor Teks: Sublinear TF-IDF + Latent Semantic Analysis (TruncatedSVD LSA).
3. Penentuan Jumlah Klaster Optimal ($K$): Evaluasi grafik **Elbow Method (Inertia)** & **Silhouette Index**.
4. Algoritma Klasterisasi: K-Means, MiniBatchKMeans, dan Agglomerative Clustering.
5. Interpretasi Topik (c-TF-IDF): Ekstraksi otomatis kata-kata kunci dominan untuk setiap cluster.
6. Evaluasi Adjusted Rand Score (ARI) & Ekspor Berkas Submission format `ID, Cluster`.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
import warnings
warnings.filterwarnings('ignore')

SEED = 2026
print("Modul clustering dan topic modeling siap.")


## 1. Memuat Dataset Teks Tanpa Label (ID, Title, Urls)
Ubah `DATASET_PATH` ke lokasi berkas kompetisi Anda (misal `cases/web_mining/data.csv`). Jika belum ada, pipeline membuat data simulasi berita trending.


In [ ]:
DATASET_PATH = "web_mining_news.csv"

if not os.path.exists(DATASET_PATH):
    print("Membuat dataset sampel berita trending kompas...")
    sample_news = {
        "ID": range(1, 21),
        "Title": [
            # Topik 0: Sepak Bola / Timnas
            "Timnas U23 Indonesia Lolos Semifinal Piala Asia Usai Kalahkan Korea",
            "Jadwal Lengkap Pertandingan Timnas Indonesia di Piala Asia U23 Qatar",
            "Shin Tae-yong Puji Mental Pemain Timnas Usai Laga Sengit Adu Penalti",
            "Profil Wasit Laga Timnas Indonesia vs Korea Selatan di Perempat Final",
            # Topik 1: Arus Mudik / Lebaran
            "Puncak Arus Mudik Lebaran 2024 Jalur Tol Trans Jawa Mulai Padat",
            "Jadwal Lengkap Rekayasa One Way dan Contraflow Tol Trans Jawa",
            "Kakorlantas Imbau Pemudik Tidak Istirahat di Bahu Jalan Tol",
            "Pantauan Arus Balik H+3 Lebaran Pelabuhan Bakauheni Mulai Ramai",
            # Topik 2: Ekonomi / Rupiah / IHSG
            "Rupiah Melemah Terhadap Dollar AS Dampak Konflik Geopolitik Global",
            "IHSG Diprediksi Menguat Pasca Libur Lebaran Cek Saham Pilihan",
            "Harga Emas Antam Naik Tajam Hari Ini Simak Rincian Terbarunya",
            "Bank Indonesia Siapkan Langkah Intervensi Stabilkan Nilai Tukar Rupiah",
            # Topik 3: Teknologi / Gadget / AI
            "Apple Rilis Pembaruan iOS Terbaru untuk Atasi Bug Keamanan",
            "Perkembangan AI Generatif Berpotensi Ubah Lanskap Industri Teknologi",
            "Spesifikasi dan Harga Smartphone Terbaru yang Rilis Bulan Ini",
            "Google Uji Fitur Pencarian Berbasis Kecerdasan Buatan Generative AI",
            # Topik 4: Bencana / Cuaca BMKG
            "BMKG Keluarkan Peringatan Dini Cuaca Ekstrem Hujan Lebat di Wilayah Ini",
            "Dampak Gempa Bumi Magnitudo 6.2 di Wilayah Pesisir Barat Sumatera",
            "Prakiraan Cuaca Jabodetabek Hari Ini Berpotensi Hujan Disertai Angin",
            "Status Aktivitas Gunung Berapi Meningkat Warga Diminta Jauhi Zona Bahaya"
        ],
        "Urls": [f"https://kompas.com/read/2024/04/{i}" for i in range(1, 21)]
    }
    df = pd.DataFrame(sample_news)
else:
    df = pd.read_csv(DATASET_PATH)

print(f"Total Dokumen Berita: {len(df)}")
print(df.head())


## 2. Text Preprocessing Khusus Judul Berita


In [ ]:
STOPWORDS_ID = set([
    "di", "ke", "dari", "dan", "yang", "untuk", "ini", "itu", "atau", "pada",
    "dengan", "adalah", "sebagai", "dalam", "bisa", "usai", "hari", "terbaru",
    "pasca", "simak", "cek", "mulai", "terhadap", "akan", "oleh", "para"
])

def preprocess_news_title(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    words = [w for w in text.split() if w not in STOPWORDS_ID and len(w) > 2]
    return " ".join(words)

df["clean_title"] = df["Title"].apply(preprocess_news_title)
print("Contoh Hasil Pembersihan:")
for i in range(min(3, len(df))):
    print("ASLI   :", df["Title"].iloc[i])
    print("BERSIH :", df["clean_title"].iloc[i])


## 3. Ekstraksi Fitur TF-IDF & Reduksi Dimensi (LSA)


In [ ]:
# Representasi TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(df["clean_title"])

# Reduksi Dimensi LSA (Latent Semantic Analysis) untuk membuang noise
n_components = min(50, X_tfidf.shape[1] - 1, len(df) - 1)
svd = TruncatedSVD(n_components=n_components, random_state=SEED)
X_dense = svd.fit_transform(X_tfidf)

print(f"Matriks TF-IDF: {X_tfidf.shape} -> Matriks LSA Terkompresi: {X_dense.shape}")


## 4. Pencarian Jumlah Klaster Optimal ($K$): Elbow Curve & Silhouette Score


In [ ]:
max_k = min(10, len(df) - 1)
k_range = range(2, max_k + 1)
inertias = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    cluster_labels = km.fit_predict(X_dense)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_dense, cluster_labels)
    silhouette_scores.append(sil)
    print(f"K = {k:2d} | Silhouette Score: {sil:.4f} | Inertia: {km.inertia_:.2f}")

# Visualisasi Elbow & Silhouette
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(k_range, inertias, 'bo-', label="Inertia (Elbow)")
ax1.set_xlabel("Jumlah Cluster (K)")
ax1.set_ylabel("Inertia", color='b')

ax2 = ax1.twinx()
ax2.plot(k_range, silhouette_scores, 'ro--', label="Silhouette Score")
ax2.set_ylabel("Silhouette Score", color='r')
plt.title("Optimasi Pemilihan Jumlah Klaster (Elbow Method & Silhouette Index)")
plt.grid(True, alpha=0.3)
plt.show()

best_k = k_range[np.argmax(silhouette_scores)]
print(f"\n>>> Rekomendasi K Terbaik Berdasarkan Silhouette Score: K = {best_k} (Skor: {max(silhouette_scores):.4f}) <<<")


## 5. Eksekusi Klasterisasi Akhir (K-Means / Agglomerative)


In [ ]:
CHOSEN_K = best_k  # atau ganti manual jika Anda menganalisis elbow curve

model_clustering = KMeans(n_clusters=CHOSEN_K, random_state=SEED, n_init=20)
df["Cluster"] = model_clustering.fit_predict(X_dense)

print(f"Distribusi Dokumen per Klaster (K={CHOSEN_K}):")
print(df["Cluster"].value_counts().sort_index())


## 6. Interpretasi Topik: Kata Kunci Dominan per Klaster (c-TF-IDF)


In [ ]:
terms = np.array(tfidf.get_feature_names_out())
print("=== KATA KUNCI DOMINAN TIAP KLASTER TOPIK ===")

for c in range(CHOSEN_K):
    cluster_docs = df[df["Cluster"] == c]
    if len(cluster_docs) == 0:
        continue
    c_text = " ".join(cluster_docs["clean_title"])
    c_vec = tfidf.transform([c_text]).toarray()[0]
    top_indices = np.argsort(c_vec)[-8:][::-1]
    top_words = terms[top_indices]
    
    print(f"\n[Klaster {c}] ({len(cluster_docs)} Berita):")
    print("Kata Kunci:", ", ".join(top_words))
    print("Contoh Judul:")
    for t in cluster_docs["Title"].head(2):
        print(f"  - {t}")


## 7. Pembuatan Berkas Hasil Akhir (submission.csv)
Format kolom: `ID, Cluster` (sesuai spesifikasi kompetisi Kaggle).


In [ ]:
submission_df = df[["ID", "Cluster"]].copy()
submission_df.to_csv("submission_web_mining_clusters.csv", index=False)

print("Berkas 'submission_web_mining_clusters.csv' berhasil dibuat:")
print(submission_df.head(10))
